### Import Dependencies

In [37]:
import os
import openai
from qdrant_client import QdrantClient
from langsmith import Client

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

### Download an example reference data point from LangSmith

In [38]:
from dotenv import load_dotenv

load_dotenv("../../.env")

True

In [39]:
ls_client = Client()

In [40]:
dataset = ls_client.read_dataset(
    dataset_name="coordinator-delegation-evaluation"
)

In [41]:
dataset

Dataset(name='coordinator-delegation-evaluation', description='', data_type=<DataType.kv: 'kv'>, id=UUID('50f756cd-bbab-4cd7-b248-77ca9380f493'), created_at=datetime.datetime(2026, 9, 11, 2, 43, 13, 993198, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 9, 11, 2, 43, 13, 993198, tzinfo=TzInfo(0)), example_count=3, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None, metadata=None)

In [42]:
list(ls_client.list_examples(dataset_id=dataset.id, limit=50))

[<class 'langsmith.schemas.Example'>(id=fbe8867f-ce0e-4f73-8f34-7ca2821c8b50, dataset_id=50f756cd-bbab-4cd7-b248-77ca9380f493, link='https://smith.langchain.com/o/fea55b69-d19d-491c-810a-57660f3b2c4c/datasets/50f756cd-bbab-4cd7-b248-77ca9380f493/e/fbe8867f-ce0e-4f73-8f34-7ca2821c8b50'),
 <class 'langsmith.schemas.Example'>(id=118a7109-01d1-46b2-bb51-d92970e8dd33, dataset_id=50f756cd-bbab-4cd7-b248-77ca9380f493, link='https://smith.langchain.com/o/fea55b69-d19d-491c-810a-57660f3b2c4c/datasets/50f756cd-bbab-4cd7-b248-77ca9380f493/e/118a7109-01d1-46b2-bb51-d92970e8dd33'),
 <class 'langsmith.schemas.Example'>(id=b5c1f746-765d-4b55-b2c7-4fbf78b8e8d6, dataset_id=50f756cd-bbab-4cd7-b248-77ca9380f493, link='https://smith.langchain.com/o/fea55b69-d19d-491c-810a-57660f3b2c4c/datasets/50f756cd-bbab-4cd7-b248-77ca9380f493/e/b5c1f746-765d-4b55-b2c7-4fbf78b8e8d6')]

In [43]:
reference_inputs = [item.inputs["input"] for item in list(ls_client.list_examples(dataset_id=dataset.id, limit=50))]

In [44]:
reference_inputs

[{'messages': [{'type': 'human',
    'content': 'Can i get some earphones and a laptop? Could you also give me some positive and negative reviews about each suggestion? Can you then add the best reviewed items from each category to my cart and reserve them in your warehouses?',
    'additional_kwargs': {},
    'response_metadata': {}},
   {'id': 'resp_0aa3f66d020bcc6d006aa36c3de73887d2ba8b7be5fc2f93d8',
    'type': 'ai',
    'content': [{'id': 'rs_0aa3f66d020bcc6d006aa36c3e7a7487d2b002f39f5b2737d6',
      'type': 'reasoning',
      'content': [],
      'summary': [],
      'encrypted_content': 'gAAAAABqo2w_8c1KMpq8ns01kAyP9YnXddlsKcF0ASJGtr0SiMjKm2HVAbxYc_86mbhGh65bY8aXDL0jGsJzMZeOXDqqba0NQd2UQRz_zFjoZE6Y_um633xA1UL6xfTP2igBhPX4gSAN2rtPiiHzTrKrhpqKIcILqgSKMIPlemoJcmjg82CHepXyfwUqRlPcGsp3JzDvoDEk3km2WedHYGFVAlwIv1nX2g1XKR8ZwN5912EDRw64o1PRTmtgCSVrBtSr_YRpsHiY6pWQJpc13y1pKVVPTO2kyy1Zs3COkFfjb_qrJk7DMVp4BeHNZEHJEB_TvDzsmA2cNzW6beojILE_nhK2j83UnbcCcgwgX0UbRRf_trFmTiOXHEDxKnlyReTvS7O-kRzms-

In [45]:
reference_outputs = [item.outputs for item in list(ls_client.list_examples(dataset_id=dataset.id, limit=50))]

In [46]:
reference_outputs

[{'answer': '',
  'coordinator_agent': {'plan': [{'task': 'Reserve the selected earphones and laptop in the warehouses after they have been added to the cart.',
     'agent': 'warehouse_manager_agent'}],
   'next_agent': 'warehouse_manager_agent',
   'final_answer': False}},
 {'answer': '',
  'coordinator_agent': {'plan': [], 'next_agent': '', 'final_answer': True}},
 {'answer': '',
  'coordinator_agent': {'plan': [], 'next_agent': '', 'final_answer': True}}]

### Coordinator Agent Evaluation

In [47]:
from pydantic import BaseModel

from langchain_openai import ChatOpenAI

from langsmith import traceable

from langchain_core.messages import SystemMessage, AIMessage
from IPython.display import display

from typing import Any, Annotated, List
from pydantic import Field
from operator import add

from jinja2 import Template

In [48]:
class Delegation(BaseModel):
    agent: str = Field(description="The agent to delegate the task to.")
    task: str = Field(description="The task to be performed by the agent.")

class Plan(BaseModel):
    next_agent: str = Field(description="The next agent to invoke")
    plan: List[Delegation] = Field(description="A list of delegations to agents with tasks to be performed in sequence.")

class FinalAgentResponse(BaseModel):

    answer: str = Field(description="Answer to the question")

class CoordinatorAgentProperties(BaseModel):
    iteration: int = 0
    final_answer: bool = False
    plan: List[Delegation] = []
    next_agent: str = ""
    
class State(BaseModel):
    messages: Annotated[List[Any], add] = []
    coordinator_agent: CoordinatorAgentProperties = CoordinatorAgentProperties()
    answer: str = ""

In [49]:
@traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-mini"
    }
)
def coordinator_agent(state) -> dict:
    
    prompt_template = """You are a Coordinator Agent as part of a shopping assistant.

## Instructions

- Your role is to create plans for solving user queries and delegate the tasks accordingly.
- You will be given a conversation history, your task is to create a plan for solving the user's query.
- After the plan is created, you should output the next agent to invoke and the task to be performed by that agent.
- Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and revise the plan.
- If there is a sequence of tasks to be performed by a single agent, you should combine them into a single task.
- Do not route to any agent if the user's query needs clarification or is irelevant. Do it yourself.

## Available Agents

- product_qna_agent: The user is asking a question about a product. This can be a question about available products, their specifications, user reviews etc.
- shopping_cart_agent: The user is asking to add or remove items from the shopping cart or questions about the current shopping cart.
- warehouse_manager_agent: The user is asking to reserve items from the warehouses or about availability of the items in warehouses.

## Examples

Question: "Do you have running shoes under $100?"
Next agent: product_qna_agent

Question: "Can you list the items in my cart?"
Next agent: shopping_cart_agent

Question: "Can you reserve my shopping cart?"
Next agent: warehouse_manager_agent
"""

    template = Template(prompt_template)

    prompt = template.render()

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort="low",
        use_responses_api=True
    )
    llm_with_tools = llm.bind_tools(
        [FinalAgentResponse, Plan],
        tool_choice="required"
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages
        ]
    )

    final_answer = False
    answer = ""
    plan = []
    next_agent = ""

    def sanitise_response(response):

        for tool_call in response.tool_calls:
            if tool_call.get("name") == "FinalAgentResponse":
                answer = tool_call.get("args").get("answer")

        return AIMessage(content=answer)

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get("name") == "Plan":
            plan = response.tool_calls[0].get("args").get("plan")
            next_agent = response.tool_calls[0].get("args").get("next_agent")
            response = None
        else:
            for tool_call in response.tool_calls:
                if tool_call.get("name") == "FinalAgentResponse":
                    final_answer = True
                    answer = tool_call.get("args").get("answer")

                    response = sanitise_response(response)

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "iteration": state.coordinator_agent.iteration + 1,
            "final_answer": final_answer,
            "plan": plan,
            "next_agent": next_agent
        },
        "answer": answer
    }

In [50]:
output = coordinator_agent(
    State(
        messages=reference_inputs[0]["messages"],
        coordinator_agent=CoordinatorAgentProperties(
            iteration=0,
            final_answer=False,
            plan=[],
            next_agent=""
        ),
        answer=""
    )
)

In [51]:
output

{'messages': [],
 'coordinator_agent': {'iteration': 1,
  'final_answer': False,
  'plan': [{'agent': 'shopping_cart_agent',
    'task': 'Add the best-reviewed earphones and laptop to the user\'s cart. The selected items are Betron BS10 Earphones Wired (ID: B0BXT863XV) and HP 17-BY1053DX 17.3" Laptop (ID: B07Y5LZBK4).'},
   {'agent': 'warehouse_manager_agent',
    'task': 'Reserve the same two items from the warehouses after they have been added to the cart.'}],
  'next_agent': 'shopping_cart_agent'},
 'answer': ''}

In [52]:
def evaluate_coordinator_delegation(run, example):

    final_answer_match = run["coordinator_agent"]["final_answer"] == example["coordinator_agent"]["final_answer"]
    next_agent_match = run["coordinator_agent"]["next_agent"] == example["coordinator_agent"]["next_agent"]

    return final_answer_match and next_agent_match

In [53]:
evaluate_coordinator_delegation(output, reference_outputs[0])

False

### Run against LangSmith

In [54]:
def evaluate_coordinator_delegation(run, example):

    final_answer_match = run.outputs["coordinator_agent"]["final_answer"] == example.outputs["coordinator_agent"]["final_answer"]
    next_agent_match = run.outputs["coordinator_agent"]["next_agent"] == example.outputs["coordinator_agent"]["next_agent"]

    return final_answer_match and next_agent_match

In [55]:
results = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x["input"]["messages"],
            coordinator_agent=CoordinatorAgentProperties(
                iteration=0,
                final_answer=False,
                plan=[],
                next_agent=""
            ),
            answer=""
        )
    ),
    data="coordinator-delegation-evaluation",
    evaluators=[
        evaluate_coordinator_delegation
    ],
    experiment_prefix="coordinator-delegation",
    max_concurrency=10,
    num_repetitions=5
)

View the evaluation results for experiment: 'coordinator-delegation-163b3d79' at:
https://smith.langchain.com/o/fea55b69-d19d-491c-810a-57660f3b2c4c/datasets/50f756cd-bbab-4cd7-b248-77ca9380f493/compare?selectedSessions=e5f3069f-9417-4009-8506-a35271f7be67




15it [00:05,  2.58it/s]


### Coordinator Agegnt V1

In [56]:
class Delegation(BaseModel):
    agent: str = Field(description="The agent to delegate the task to.")
    task: str = Field(description="The task to be performed by the agent.")

class Plan(BaseModel):
    next_agent: str = Field(description="The next agent to invoke")
    plan: List[Delegation] = Field(description="A list of delegations to agents with tasks to be performed in sequence.")

class FinalAgentResponse(BaseModel):

    answer: str = Field(description="Answer to the question")

class CoordinatorAgentProperties(BaseModel):
    iteration: int = 0
    final_answer: bool = False
    plan: List[Delegation] = []
    next_agent: str = ""
    
class State(BaseModel):
    messages: Annotated[List[Any], add] = []
    coordinator_agent: CoordinatorAgentProperties = CoordinatorAgentProperties()
    answer: str = ""

In [57]:
@traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-mini"
    }
)
def coordinator_agent(state) -> dict:
    
    prompt_template = """You are a Coordinator Agent as part of a shopping assistant.

## Instructions

- Your role is to delegate work to worker agents in order to solve user queries.
- You should output the next agent to invoke.
- Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and output the next delegation via the `Plan` tool or output the final answer via the `FinalAgentResponse` tool.
- Do not route to any agent if the user's query needs clarification or is irelevant. Do it yourself.
- Only route to the `warehouse_manager_agent` if the user has confirmed that they want to reserve an item and it was already successfully added to the shopping cart.
- Only route to the `shopping_cart_agent` if the user specifically asks about their shopping cart or wants to add or remove items from the shopping cart.
- Do not delegate work to the same agent twice in a row.

## Routing rules

Match the user's latest message to one of these patterns:

1. PRODUCT QUESTION — asks about products, specs, reviews, availability, suggestions.
    Examples: "can I get some earphones", "show me laptops", "what watches do you have", "tell me about X", "is X any good", "I'd like to see some X".
    → Route ONLY to `product_qna_agent`. When it returns, emit `FinalAgentResponse`. Do NOT chain to cart or warehouse.

2. EXPLICIT CART INSTRUCTION — uses cart vocabulary.
    Examples: "add to my cart", "remove from my cart", "what's in my cart", "put X in my cart", "delete X from cart".
    → Route to `shopping_cart_agent`. (If items must be discovered first, route to `product_qna_agent` first.)

3. EXPLICIT RESERVATION INSTRUCTION — uses warehouse/reservation vocabulary.
    Examples: "reserve these", "hold these for me", "check warehouse stock for X".
    → Route to `warehouse_manager_agent` ONLY AFTER items are confirmed in the cart.

Default rule: if the user's message does NOT contain explicit cart or warehouse vocabulary, do NOT delegate to those agents. Phrases like "can I get", "I want",
"I'd like", "show me", "I need" are PRODUCT QUESTIONS, not purchase instructions. The user must explicitly say "add", "cart", "reserve", "buy", or "order" before those agents are in scope.

## Available Agents

- product_qna_agent: The user is asking a question about a product. This can be a question about available products, their specifications, user reviews etc.
- shopping_cart_agent: The user is asking to add or remove items from the shopping cart or questions about the current shopping cart.
- warehouse_manager_agent: The user is asking to reserve items from the warehouses or about availability of the items in warehouses.
"""

    template = Template(prompt_template)

    prompt = template.render()

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort="low",
        use_responses_api=True
    )
    llm_with_tools = llm.bind_tools(
        [FinalAgentResponse, Plan],
        tool_choice="required"
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages
        ]
    )

    final_answer = False
    answer = ""
    plan = []
    next_agent = ""

    def sanitise_response(response):

        for tool_call in response.tool_calls:
            if tool_call.get("name") == "FinalAgentResponse":
                answer = tool_call.get("args").get("answer")

        return AIMessage(content=answer)

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get("name") == "Plan":
            plan = response.tool_calls[0].get("args").get("plan")
            next_agent = response.tool_calls[0].get("args").get("next_agent")
            response = None
        else:
            for tool_call in response.tool_calls:
                if tool_call.get("name") == "FinalAgentResponse":
                    final_answer = True
                    answer = tool_call.get("args").get("answer")

                    response = sanitise_response(response)

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "iteration": state.coordinator_agent.iteration + 1,
            "final_answer": final_answer,
            "plan": plan,
            "next_agent": next_agent
        },
        "answer": answer
    }

In [58]:
results = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x["input"]["messages"],
            coordinator_agent=CoordinatorAgentProperties(
                iteration=0,
                final_answer=False,
                plan=[],
                next_agent=""
            ),
            answer=""
        )
    ),
    data="coordinator-delegation-evaluation",
    evaluators=[
        evaluate_coordinator_delegation
    ],
    experiment_prefix="coordinator-delegation",
    max_concurrency=10,
    num_repetitions=5
)

View the evaluation results for experiment: 'coordinator-delegation-9fa7ef81' at:
https://smith.langchain.com/o/fea55b69-d19d-491c-810a-57660f3b2c4c/datasets/50f756cd-bbab-4cd7-b248-77ca9380f493/compare?selectedSessions=7a331398-b283-46f2-8fbe-1938fc23c54a




15it [00:05,  2.72it/s]


### Coordinator Agegnt V2

In [59]:
class Delegation(BaseModel):
    agent: str = Field(description="The agent to delegate the task to.")
    task: str = Field(description="The task to be performed by the agent.")

class Plan(BaseModel):
    next_agent: str = Field(description="The next agent to invoke")
    plan: List[Delegation] = Field(description="A list of delegations to agents with tasks to be performed in sequence.")

class FinalAgentResponse(BaseModel):

    answer: str = Field(description="Answer to the question")

class CoordinatorAgentProperties(BaseModel):
    iteration: int = 0
    final_answer: bool = False
    plan: List[Delegation] = []
    next_agent: str = ""
    
class State(BaseModel):
    messages: Annotated[List[Any], add] = []
    coordinator_agent: CoordinatorAgentProperties = CoordinatorAgentProperties()
    answer: str = ""

In [60]:
@traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-mini"
    }
)
def coordinator_agent(state) -> dict:
    
    prompt_template = """You are a Coordinator Agent as part of a shopping assistant.

## Instructions

- Your role is to delegate work to worker agents in order to solve user queries.
- You should output the next agent to invoke.
- Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and output the next delegation via the `Plan` tool.
- If you do not need to delegate work to any agent, output the final answer via the `FinalAgentResponse` tool.
- Do not route to any agent if the user's query needs clarification or is irelevant. Do it yourself.
- Only route to the `warehouse_manager_agent` if the user has confirmed that they want to reserve an item and it was already successfully added to the shopping cart.
- Only route to the `shopping_cart_agent` if the user specifically asks about their shopping cart or wants to add or remove items from the shopping cart.
- Do not delegate work to the same agent twice in a row.

## Routing rules

Match the user's latest message to one of these patterns:

1. PRODUCT QUESTION — asks about products, specs, reviews, availability, suggestions.
    Examples: "can I get some earphones", "show me laptops", "what watches do you have", "tell me about X", "is X any good", "I'd like to see some X".
    → Route ONLY to `product_qna_agent`. When it returns, emit `FinalAgentResponse`. Do NOT chain to cart or warehouse.

2. EXPLICIT CART INSTRUCTION — uses cart vocabulary.
    Examples: "add to my cart", "remove from my cart", "what's in my cart", "put X in my cart", "delete X from cart".
    → Route to `shopping_cart_agent`. (If items must be discovered first, route to `product_qna_agent` first.)

3. EXPLICIT RESERVATION INSTRUCTION — uses warehouse/reservation vocabulary.
    Examples: "reserve these", "hold these for me", "check warehouse stock for X".
    → Route to `warehouse_manager_agent` ONLY AFTER items are confirmed in the cart.

Default rule: if the user's message does NOT contain explicit cart or warehouse vocabulary, do NOT delegate to those agents. Phrases like "can I get", "I want",
"I'd like", "show me", "I need" are PRODUCT QUESTIONS, not purchase instructions. The user must explicitly say "add", "cart", "reserve", "buy", or "order" before those agents are in scope.

## Available Agents

- product_qna_agent: The user is asking a question about a product. This can be a question about available products, their specifications, user reviews etc.
- shopping_cart_agent: The user is asking to add or remove items from the shopping cart or questions about the current shopping cart.
- warehouse_manager_agent: The user is asking to reserve items from the warehouses or about availability of the items in warehouses.
"""

    template = Template(prompt_template)

    prompt = template.render()

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort="low",
        use_responses_api=True
    )
    llm_with_tools = llm.bind_tools(
        [FinalAgentResponse, Plan],
        tool_choice="required"
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages
        ]
    )

    final_answer = False
    answer = ""
    plan = []
    next_agent = ""

    def sanitise_response(response):

        for tool_call in response.tool_calls:
            if tool_call.get("name") == "FinalAgentResponse":
                answer = tool_call.get("args").get("answer")

        return AIMessage(content=answer)

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get("name") == "Plan":
            plan = response.tool_calls[0].get("args").get("plan")
            next_agent = response.tool_calls[0].get("args").get("next_agent")
            response = None
        else:
            for tool_call in response.tool_calls:
                if tool_call.get("name") == "FinalAgentResponse":
                    final_answer = True
                    answer = tool_call.get("args").get("answer")

                    response = sanitise_response(response)

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "iteration": state.coordinator_agent.iteration + 1,
            "final_answer": final_answer,
            "plan": plan,
            "next_agent": next_agent
        },
        "answer": answer
    }

In [61]:
results = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x["input"]["messages"],
            coordinator_agent=CoordinatorAgentProperties(
                iteration=0,
                final_answer=False,
                plan=[],
                next_agent=""
            ),
            answer=""
        )
    ),
    data="coordinator-delegation-evaluation",
    evaluators=[
        evaluate_coordinator_delegation
    ],
    experiment_prefix="coordinator-delegation",
    max_concurrency=10,
    num_repetitions=5
)

View the evaluation results for experiment: 'coordinator-delegation-9f534252' at:
https://smith.langchain.com/o/fea55b69-d19d-491c-810a-57660f3b2c4c/datasets/50f756cd-bbab-4cd7-b248-77ca9380f493/compare?selectedSessions=19c43d14-34d7-4058-ac4b-2f85ca8b4957




11it [00:05,  2.33it/s]Error running target function: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-5.4-mini in organization org-2noS17r28MSZKa1chd6e7dxG on tokens per min (TPM): Limit 200000, Used 198182, Requested 4335. Please try again in 755ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}
Traceback (most recent call last):
  File "/Users/hang/Documents/Development/ai-eng-course/ai-eng-project/.venv/lib/python3.12/site-packages/langsmith/evaluation/_runner.py", line 1976, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
  File "/var/folders/wl/bfvmh_sd2_91_635wbryp9hm0000gn/T/ipykernel_11421/361648355.py", line 2, in <lambda>
    lambda x: coordinator_agent(
              ^^^^^^^^^^^^^^^^^^
  File "/var/folders/wl/bfvmh_sd2_91_635wbryp9hm0000gn/T/ipykernel_11421/3305486632.py", line 64, in coordinator_agent
    response = llm_with_tools.invoke(
               ^^

### Coordinator Agegnt V3

In [62]:
class Delegation(BaseModel):
    agent: str = Field(description="The agent to delegate the task to.")
    task: str = Field(description="The task to be performed by the agent.")

class Plan(BaseModel):
    next_agent: str = Field(description="The next agent to invoke")
    plan: List[Delegation] = Field(description="A list of delegations to agents with tasks to be performed in sequence.")

class FinalAgentResponse(BaseModel):

    answer: str = Field(description="Answer to the question")

class CoordinatorAgentProperties(BaseModel):
    iteration: int = 0
    final_answer: bool = False
    plan: List[Delegation] = []
    next_agent: str = ""
    
class State(BaseModel):
    messages: Annotated[List[Any], add] = []
    coordinator_agent: CoordinatorAgentProperties = CoordinatorAgentProperties()
    answer: str = ""

In [63]:
@traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-mini"
    }
)
def coordinator_agent(state) -> dict:
    
    prompt_template = """You are a Coordinator Agent as part of a shopping assistant.

## Instructions

- Your role is to delegate work to worker agents in order to solve user queries.
- You should output the next agent to invoke.
- Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and output the next delegation via the `Plan` tool.
- If you do not need to delegate work to any agent, output the final answer via the `FinalAgentResponse` tool.
- Do not route to any agent if the user's query needs clarification or is irelevant. Do it yourself.
- Only route to the `warehouse_manager_agent` if the user has confirmed that they want to reserve an item and it was already successfully added to the shopping cart.
- Only route to the `shopping_cart_agent` if the user specifically asks about their shopping cart or wants to add or remove items from the shopping cart.
- Do not delegate work to the same agent twice in a row.

## Routing rules

Match the user's latest message to one of these patterns:

1. PRODUCT QUESTION — asks about products, specs, reviews, availability, suggestions.
    Examples: "can I get some earphones", "show me laptops", "what watches do you have", "tell me about X", "is X any good", "I'd like to see some X".
    → Route ONLY to `product_qna_agent`. When it returns, emit `FinalAgentResponse`. Do NOT chain to cart or warehouse.

2. EXPLICIT CART INSTRUCTION — uses cart vocabulary.
    Examples: "add to my cart", "remove from my cart", "what's in my cart", "put X in my cart", "delete X from cart".
    → Route to `shopping_cart_agent`. (If items must be discovered first, route to `product_qna_agent` first.)

3. EXPLICIT RESERVATION INSTRUCTION — uses warehouse/reservation vocabulary.
    Examples: "reserve these", "hold these for me", "check warehouse stock for X".
    → Route to `warehouse_manager_agent` ONLY AFTER items are confirmed in the cart.

Default rule: if the user's message does NOT contain explicit cart or warehouse vocabulary, do NOT delegate to those agents. Phrases like "can I get", "I want",
"I'd like", "show me", "I need" are PRODUCT QUESTIONS, not purchase instructions. The user must explicitly say "add", "cart", "reserve", "buy", or "order" before those agents are in scope.

## Available Agents

- product_qna_agent: The user is asking a question about a product. This can be a question about available products, their specifications, user reviews etc.
- shopping_cart_agent: The user is asking to add or remove items from the shopping cart or questions about the current shopping cart.
- warehouse_manager_agent: The user is asking to reserve items from the warehouses or about availability of the items in warehouses.
"""

    template = Template(prompt_template)

    prompt = template.render()

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort="medium",
        use_responses_api=True
    )
    llm_with_tools = llm.bind_tools(
        [FinalAgentResponse, Plan],
        tool_choice="required"
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages
        ]
    )

    final_answer = False
    answer = ""
    plan = []
    next_agent = ""

    def sanitise_response(response):

        for tool_call in response.tool_calls:
            if tool_call.get("name") == "FinalAgentResponse":
                answer = tool_call.get("args").get("answer")

        return AIMessage(content=answer)

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get("name") == "Plan":
            plan = response.tool_calls[0].get("args").get("plan")
            next_agent = response.tool_calls[0].get("args").get("next_agent")
            response = None
        else:
            for tool_call in response.tool_calls:
                if tool_call.get("name") == "FinalAgentResponse":
                    final_answer = True
                    answer = tool_call.get("args").get("answer")

                    response = sanitise_response(response)

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "iteration": state.coordinator_agent.iteration + 1,
            "final_answer": final_answer,
            "plan": plan,
            "next_agent": next_agent
        },
        "answer": answer
    }

In [64]:
results = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x["input"]["messages"],
            coordinator_agent=CoordinatorAgentProperties(
                iteration=0,
                final_answer=False,
                plan=[],
                next_agent=""
            ),
            answer=""
        )
    ),
    data="coordinator-delegation-evaluation",
    evaluators=[
        evaluate_coordinator_delegation
    ],
    experiment_prefix="coordinator-delegation",
    max_concurrency=10,
    num_repetitions=5
)

View the evaluation results for experiment: 'coordinator-delegation-6371b9bf' at:
https://smith.langchain.com/o/fea55b69-d19d-491c-810a-57660f3b2c4c/datasets/50f756cd-bbab-4cd7-b248-77ca9380f493/compare?selectedSessions=12b653cf-fe6d-4f93-9a26-b5a12b561e36




4it [00:04,  1.23it/s]Error running target function: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-5.4-mini in organization org-2noS17r28MSZKa1chd6e7dxG on tokens per min (TPM): Limit 200000, Used 200000, Requested 4335. Please try again in 1.3s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}
Traceback (most recent call last):
  File "/Users/hang/Documents/Development/ai-eng-course/ai-eng-project/.venv/lib/python3.12/site-packages/langsmith/evaluation/_runner.py", line 1976, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
  File "/var/folders/wl/bfvmh_sd2_91_635wbryp9hm0000gn/T/ipykernel_11421/361648355.py", line 2, in <lambda>
    lambda x: coordinator_agent(
              ^^^^^^^^^^^^^^^^^^
  File "/var/folders/wl/bfvmh_sd2_91_635wbryp9hm0000gn/T/ipykernel_11421/2927677703.py", line 64, in coordinator_agent
    response = llm_with_tools.invoke(
               ^^^^

### Coordinator Agegnt V4

In [65]:
class Delegation(BaseModel):
    agent: str = Field(description="The agent to delegate the task to.")
    task: str = Field(description="The task to be performed by the agent.")

class Plan(BaseModel):
    next_agent: str = Field(description="The next agent to invoke")
    # plan: List[Delegation] = Field(description="A list of delegations to agents with tasks to be performed in sequence.")

class FinalAgentResponse(BaseModel):

    answer: str = Field(description="Answer to the question")

class CoordinatorAgentProperties(BaseModel):
    iteration: int = 0
    final_answer: bool = False
    # plan: List[Delegation] = []
    next_agent: str = ""
    
class State(BaseModel):
    messages: Annotated[List[Any], add] = []
    coordinator_agent: CoordinatorAgentProperties = CoordinatorAgentProperties()
    answer: str = ""

In [66]:
@traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-mini"
    }
)
def coordinator_agent(state) -> dict:
    
    prompt_template = """You are a Coordinator Agent as part of a shopping assistant.

## Instructions

- Your role is to delegate work to worker agents in order to solve user queries.
- You should output the next agent to invoke.
- Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and output the next delegation via the `Plan` tool.
- If you do not need to delegate work to any agent, output the final answer via the `FinalAgentResponse` tool.
- Do not route to any agent if the user's query needs clarification or is irelevant. Do it yourself.
- Only route to the `warehouse_manager_agent` if the user has confirmed that they want to reserve an item and it was already successfully added to the shopping cart.
- Only route to the `shopping_cart_agent` if the user specifically asks about their shopping cart or wants to add or remove items from the shopping cart.
- Do not delegate work to the same agent twice in a row.

## Routing rules

Match the user's latest message to one of these patterns:

1. PRODUCT QUESTION — asks about products, specs, reviews, availability, suggestions.
    Examples: "can I get some earphones", "show me laptops", "what watches do you have", "tell me about X", "is X any good", "I'd like to see some X".
    → Route ONLY to `product_qna_agent`. When it returns, emit `FinalAgentResponse`. Do NOT chain to cart or warehouse.

2. EXPLICIT CART INSTRUCTION — uses cart vocabulary.
    Examples: "add to my cart", "remove from my cart", "what's in my cart", "put X in my cart", "delete X from cart".
    → Route to `shopping_cart_agent`. (If items must be discovered first, route to `product_qna_agent` first.)

3. EXPLICIT RESERVATION INSTRUCTION — uses warehouse/reservation vocabulary.
    Examples: "reserve these", "hold these for me", "check warehouse stock for X".
    → Route to `warehouse_manager_agent` ONLY AFTER items are confirmed in the cart.

Default rule: if the user's message does NOT contain explicit cart or warehouse vocabulary, do NOT delegate to those agents. Phrases like "can I get", "I want",
"I'd like", "show me", "I need" are PRODUCT QUESTIONS, not purchase instructions. The user must explicitly say "add", "cart", "reserve", "buy", or "order" before those agents are in scope.

## Available Agents

- product_qna_agent: The user is asking a question about a product. This can be a question about available products, their specifications, user reviews etc.
- shopping_cart_agent: The user is asking to add or remove items from the shopping cart or questions about the current shopping cart.
- warehouse_manager_agent: The user is asking to reserve items from the warehouses or about availability of the items in warehouses.
"""

    template = Template(prompt_template)

    prompt = template.render()

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort="medium",
        use_responses_api=True
    )
    llm_with_tools = llm.bind_tools(
        [FinalAgentResponse, Plan],
        tool_choice="required"
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages
        ]
    )

    final_answer = False
    answer = ""
    # plan = []
    next_agent = ""

    def sanitise_response(response):

        for tool_call in response.tool_calls:
            if tool_call.get("name") == "FinalAgentResponse":
                answer = tool_call.get("args").get("answer")

        return AIMessage(content=answer)

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get("name") == "Plan":
            # plan = response.tool_calls[0].get("args").get("plan")
            next_agent = response.tool_calls[0].get("args").get("next_agent")
            response = None
        else:
            for tool_call in response.tool_calls:
                if tool_call.get("name") == "FinalAgentResponse":
                    final_answer = True
                    answer = tool_call.get("args").get("answer")

                    response = sanitise_response(response)

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "iteration": state.coordinator_agent.iteration + 1,
            "final_answer": final_answer,
            # "plan": plan,
            "next_agent": next_agent
        },
        "answer": answer
    }

In [67]:
results = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x["input"]["messages"],
            coordinator_agent=CoordinatorAgentProperties(
                iteration=0,
                final_answer=False,
                plan=[],
                next_agent=""
            ),
            answer=""
        )
    ),
    data="coordinator-delegation-evaluation",
    evaluators=[
        evaluate_coordinator_delegation
    ],
    experiment_prefix="coordinator-delegation",
    max_concurrency=10,
    num_repetitions=5
)

View the evaluation results for experiment: 'coordinator-delegation-5324ef8b' at:
https://smith.langchain.com/o/fea55b69-d19d-491c-810a-57660f3b2c4c/datasets/50f756cd-bbab-4cd7-b248-77ca9380f493/compare?selectedSessions=889ca30e-3ea5-4066-9768-1e85d4242861




7it [00:05,  1.62it/s]Error running target function: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-5.4-mini in organization org-2noS17r28MSZKa1chd6e7dxG on tokens per min (TPM): Limit 200000, Used 200000, Requested 9866. Please try again in 2.959s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}
Traceback (most recent call last):
  File "/Users/hang/Documents/Development/ai-eng-course/ai-eng-project/.venv/lib/python3.12/site-packages/langsmith/evaluation/_runner.py", line 1976, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
  File "/var/folders/wl/bfvmh_sd2_91_635wbryp9hm0000gn/T/ipykernel_11421/361648355.py", line 2, in <lambda>
    lambda x: coordinator_agent(
              ^^^^^^^^^^^^^^^^^^
  File "/var/folders/wl/bfvmh_sd2_91_635wbryp9hm0000gn/T/ipykernel_11421/644877961.py", line 64, in coordinator_agent
    response = llm_with_tools.invoke(
               ^^^

### Coordinator Agegnt V5

In [68]:
class Delegation(BaseModel):
    agent: str = Field(description="The agent to delegate the task to.")
    task: str = Field(description="The task to be performed by the agent.")

class Plan(BaseModel):
    next_agent: str = Field(description="The next agent to invoke")
    # plan: List[Delegation] = Field(description="A list of delegations to agents with tasks to be performed in sequence.")

class FinalAgentResponse(BaseModel):

    answer: str = Field(description="Answer to the question")

class CoordinatorAgentProperties(BaseModel):
    iteration: int = 0
    final_answer: bool = False
    # plan: List[Delegation] = []
    next_agent: str = ""
    
class State(BaseModel):
    messages: Annotated[List[Any], add] = []
    coordinator_agent: CoordinatorAgentProperties = CoordinatorAgentProperties()
    answer: str = ""

In [69]:
@traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-mini"
    }
)
def coordinator_agent(state) -> dict:
    
    prompt_template = """You are a Coordinator Agent as part of a shopping assistant.

## Instructions

- Your role is to delegate work to worker agents in order to solve user queries.
- You should output the next agent to invoke.
- Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and output the next delegation via the `Plan` tool.
- If you do not need to delegate work to any agent, output the final answer via the `FinalAgentResponse` tool.
- Do not route to any agent if the user's query needs clarification or is irelevant. Do it yourself.
- Only route to the `warehouse_manager_agent` if the user has confirmed that they want to reserve an item and it was already successfully added to the shopping cart.
- Only route to the `shopping_cart_agent` if the user specifically asks about their shopping cart or wants to add or remove items from the shopping cart.
- Do not delegate work to the same agent twice in a row.

## Routing rules

Match the user's latest message to one of these patterns:

1. PRODUCT QUESTION — asks about products, specs, reviews, availability, suggestions.
    Examples: "can I get some earphones", "show me laptops", "what watches do you have", "tell me about X", "is X any good", "I'd like to see some X".
    → Route ONLY to `product_qna_agent`. When it returns, emit `FinalAgentResponse`. Do NOT chain to cart or warehouse.

2. EXPLICIT CART INSTRUCTION — uses cart vocabulary.
    Examples: "add to my cart", "remove from my cart", "what's in my cart", "put X in my cart", "delete X from cart".
    → Route to `shopping_cart_agent`. (If items must be discovered first, route to `product_qna_agent` first.)

3. EXPLICIT RESERVATION INSTRUCTION — uses warehouse/reservation vocabulary.
    Examples: "reserve these", "hold these for me", "check warehouse stock for X".
    → Route to `warehouse_manager_agent` ONLY AFTER items are confirmed in the cart.

Default rule: if the user's message does NOT contain explicit cart or warehouse vocabulary, do NOT delegate to those agents. Phrases like "can I get", "I want",
"I'd like", "show me", "I need" are PRODUCT QUESTIONS, not purchase instructions. The user must explicitly say "add", "cart", "reserve", "buy", or "order" before those agents are in scope.

## Available Agents

- product_qna_agent: The user is asking a question about a product. This can be a question about available products, their specifications, user reviews etc.
- shopping_cart_agent: The user is asking to add or remove items from the shopping cart or questions about the current shopping cart.
- warehouse_manager_agent: The user is asking to reserve items from the warehouses or about availability of the items in warehouses.
"""

    template = Template(prompt_template)

    prompt = template.render()

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort="medium",
        use_responses_api=True
    )
    llm_with_tools = llm.bind_tools(
        [FinalAgentResponse, Plan],
        tool_choice="required"
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages
        ]
    )

    final_answer = False
    answer = ""
    # plan = []
    next_agent = ""

    def sanitise_response(response):

        for tool_call in response.tool_calls:
            if tool_call.get("name") == "FinalAgentResponse":
                answer = tool_call.get("args").get("answer")

        return AIMessage(content=answer)

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get("name") == "Plan":
            # plan = response.tool_calls[0].get("args").get("plan")
            next_agent = response.tool_calls[0].get("args").get("next_agent")
            response = None
        else:
            for tool_call in response.tool_calls:
                if tool_call.get("name") == "FinalAgentResponse":
                    final_answer = True
                    answer = tool_call.get("args").get("answer")

                    response = sanitise_response(response)

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "iteration": state.coordinator_agent.iteration + 1,
            "final_answer": final_answer,
            # "plan": plan,
            "next_agent": next_agent
        },
        "answer": answer
    }

In [71]:
results = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x["input"]["messages"],
            coordinator_agent=CoordinatorAgentProperties(
                iteration=0,
                final_answer=False,
                plan=[],
                next_agent=""
            ),
            answer=""
        )
    ),
    data="coordinator-delegation-evaluation-2",
    evaluators=[
        evaluate_coordinator_delegation
    ],
    experiment_prefix="coordinator-delegation",
    max_concurrency=10,
    num_repetitions=5
)

View the evaluation results for experiment: 'coordinator-delegation-0ea2ebd9' at:
https://smith.langchain.com/o/fea55b69-d19d-491c-810a-57660f3b2c4c/datasets/5b0db12b-ec4c-427e-bda6-154f2ea502b0/compare?selectedSessions=d181b976-4455-40af-87b2-b6734115537a




15it [00:11,  1.28it/s]


In [75]:
results = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x["input"]["messages"],
            coordinator_agent=CoordinatorAgentProperties(
                iteration=0,
                final_answer=False,
                plan=[],
                next_agent=""
            ),
            answer=""
        )
    ),
    data="coordinator-delegation-evaluation-2",
    evaluators=[
        evaluate_coordinator_delegation
    ],
    experiment_prefix="coordinator-delegation",
    max_concurrency=10,
    num_repetitions=10
)

View the evaluation results for experiment: 'coordinator-delegation-24eb8626' at:
https://smith.langchain.com/o/fea55b69-d19d-491c-810a-57660f3b2c4c/datasets/5b0db12b-ec4c-427e-bda6-154f2ea502b0/compare?selectedSessions=3502d078-f55e-4db2-b712-ac96e97da679




30it [00:15,  1.90it/s]


#### rerun after making new changes to dataset 2

In [78]:
results = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x["input"]["messages"],
            coordinator_agent=CoordinatorAgentProperties(
                iteration=0,
                final_answer=False,
                plan=[],
                next_agent=""
            ),
            answer=""
        )
    ),
    data="coordinator-delegation-evaluation-2",
    evaluators=[
        evaluate_coordinator_delegation
    ],
    experiment_prefix="coordinator-delegation[added next agent - product_qna_agent]",
    max_concurrency=10,
    num_repetitions=10
)

View the evaluation results for experiment: 'coordinator-delegation[added next agent - product_qna_agent]-57a91c06' at:
https://smith.langchain.com/o/fea55b69-d19d-491c-810a-57660f3b2c4c/datasets/5b0db12b-ec4c-427e-bda6-154f2ea502b0/compare?selectedSessions=a47e8415-ef77-4c5a-a230-fe2f8ac52ce5




30it [00:25,  1.17it/s]


### Coordinator Agegnt V6

In [ ]:
class Delegation(BaseModel):
    agent: str = Field(description="The agent to delegate the task to.")
    task: str = Field(description="The task to be performed by the agent.")

class Plan(BaseModel):
    next_agent: str = Field(description="The next agent to invoke")
    # plan: List[Delegation] = Field(description="A list of delegations to agents with tasks to be performed in sequence.")

class FinalAgentResponse(BaseModel):

    answer: str = Field(description="Answer to the question")

class CoordinatorAgentProperties(BaseModel):
    iteration: int = 0
    final_answer: bool = False
    # plan: List[Delegation] = []
    next_agent: str = ""
    
class State(BaseModel):
    messages: Annotated[List[Any], add] = []
    coordinator_agent: CoordinatorAgentProperties = CoordinatorAgentProperties()
    answer: str = ""

In [ ]:
@traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-mini"
    }
)
def coordinator_agent(state) -> dict:
    
    prompt_template = """You are a Coordinator Agent as part of a shopping assistant.

## Instructions

- Your role is to delegate work to worker agents in order to solve user queries.
- You should output the next agent to invoke.
- Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and output the next delegation via the `Plan` tool.
- If you do not need to delegate work to any agent, output the final answer via the `FinalAgentResponse` tool.
- Do not route to any agent if the user's query needs clarification or is irelevant. Do it yourself.
- Only route to the `warehouse_manager_agent` if the user has confirmed that they want to reserve an item and it was already successfully added to the shopping cart.
- Only route to the `shopping_cart_agent` if the user specifically asks about their shopping cart or wants to add or remove items from the shopping cart.
- Do not delegate work to the same agent twice in a row.

## Routing rules

Match the user's latest message to one of these patterns:

1. PRODUCT QUESTION — asks about products, specs, reviews, availability, suggestions.
    Examples: "can I get some earphones", "show me laptops", "what watches do you have", "tell me about X", "is X any good", "I'd like to see some X".
    → Route ONLY to `product_qna_agent`. When it returns, emit `FinalAgentResponse`. Do NOT chain to cart or warehouse.

2. EXPLICIT CART INSTRUCTION — uses cart vocabulary.
    Examples: "add to my cart", "remove from my cart", "what's in my cart", "put X in my cart", "delete X from cart".
    → Route to `shopping_cart_agent`. (If items must be discovered first, route to `product_qna_agent` first.)

3. EXPLICIT RESERVATION INSTRUCTION — uses warehouse/reservation vocabulary.
    Examples: "reserve these", "hold these for me", "check warehouse stock for X".
    → Route to `warehouse_manager_agent` ONLY AFTER items are confirmed in the cart.

Default rule: if the user's message does NOT contain explicit cart or warehouse vocabulary, do NOT delegate to those agents. Phrases like "can I get", "I want",
"I'd like", "show me", "I need" are PRODUCT QUESTIONS, not purchase instructions. The user must explicitly say "add", "cart", "reserve", "buy", or "order" before those agents are in scope.

## Available Agents

- product_qna_agent: The user is asking a question about a product. This can be a question about available products, their specifications, user reviews etc.
- shopping_cart_agent: The user is asking to add or remove items from the shopping cart or questions about the current shopping cart.
- warehouse_manager_agent: The user is asking to reserve items from the warehouses or about availability of the items in warehouses.
"""

    template = Template(prompt_template)

    prompt = template.render()

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort="low",
        use_responses_api=True
    )
    llm_with_tools = llm.bind_tools(
        [FinalAgentResponse, Plan],
        tool_choice="required"
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages
        ]
    )

    final_answer = False
    answer = ""
    # plan = []
    next_agent = ""

    def sanitise_response(response):

        for tool_call in response.tool_calls:
            if tool_call.get("name") == "FinalAgentResponse":
                answer = tool_call.get("args").get("answer")

        return AIMessage(content=answer)

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get("name") == "Plan":
            # plan = response.tool_calls[0].get("args").get("plan")
            next_agent = response.tool_calls[0].get("args").get("next_agent")
            response = None
        else:
            for tool_call in response.tool_calls:
                if tool_call.get("name") == "FinalAgentResponse":
                    final_answer = True
                    answer = tool_call.get("args").get("answer")

                    response = sanitise_response(response)

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "iteration": state.coordinator_agent.iteration + 1,
            "final_answer": final_answer,
            # "plan": plan,
            "next_agent": next_agent
        },
        "answer": answer
    }

In [ ]:
results = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x["input"]["messages"],
            coordinator_agent=CoordinatorAgentProperties(
                iteration=0,
                final_answer=False,
                plan=[],
                next_agent=""
            ),
            answer=""
        )
    ),
    data="coordinator-delegation-evaluation-2",
    evaluators=[
        evaluate_coordinator_delegation
    ],
    experiment_prefix="coordinator-delegation[low reasoning effort]",
    max_concurrency=10,
    num_repetitions=10
)

### Coordinator Agegnt V7

In [79]:
class Delegation(BaseModel):
    agent: str = Field(description="The agent to delegate the task to.")
    task: str = Field(description="The task to be performed by the agent.")

class Plan(BaseModel):
    next_agent: str = Field(description="The next agent to invoke")
    # plan: List[Delegation] = Field(description="A list of delegations to agents with tasks to be performed in sequence.")

class FinalAgentResponse(BaseModel):

    answer: str = Field(description="Answer to the question")

class CoordinatorAgentProperties(BaseModel):
    iteration: int = 0
    final_answer: bool = False
    # plan: List[Delegation] = []
    next_agent: str = ""
    
class State(BaseModel):
    messages: Annotated[List[Any], add] = []
    coordinator_agent: CoordinatorAgentProperties = CoordinatorAgentProperties()
    answer: str = ""

In [80]:
@traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-mini"
    }
)
def coordinator_agent(state) -> dict:
    
    prompt_template = """You are a Coordinator Agent as part of a shopping assistant.

## Instructions

- Your role is to delegate work to worker agents in order to solve user queries.
- You should output the next agent to invoke.
- Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and output the next delegation via the `Plan` tool.
- If you do not need to delegate work to any agent, output the final answer via the `FinalAgentResponse` tool.
- Do not route to any agent if the user's query needs clarification or is irelevant. Do it yourself.
- Only route to the `warehouse_manager_agent` if the user has confirmed that they want to reserve an item and it was already successfully added to the shopping cart.
- Only route to the `shopping_cart_agent` if the user specifically asks about their shopping cart or wants to add or remove items from the shopping cart.
- Do not delegate work to the same agent twice in a row.

## Routing rules

Match the user's latest message to one of these patterns:

1. PRODUCT QUESTION — asks about products, specs, reviews, availability, suggestions.
    Examples: "can I get some earphones", "show me laptops", "what watches do you have", "tell me about X", "is X any good", "I'd like to see some X".
    → Route ONLY to `product_qna_agent`. When it returns, emit `FinalAgentResponse`. Do NOT chain to cart or warehouse.

2. EXPLICIT CART INSTRUCTION — uses cart vocabulary.
    Examples: "add to my cart", "remove from my cart", "what's in my cart", "put X in my cart", "delete X from cart".
    → Route to `shopping_cart_agent`. (If items must be discovered first, route to `product_qna_agent` first.)

3. EXPLICIT RESERVATION INSTRUCTION — uses warehouse/reservation vocabulary.
    Examples: "reserve these", "hold these for me", "check warehouse stock for X".
    → Route to `warehouse_manager_agent` ONLY AFTER items are confirmed in the cart.

Default rule: if the user's message does NOT contain explicit cart or warehouse vocabulary, do NOT delegate to those agents. Phrases like "can I get", "I want",
"I'd like", "show me", "I need" are PRODUCT QUESTIONS, not purchase instructions. The user must explicitly say "add", "cart", "reserve", "buy", or "order" before those agents are in scope.

## Available Agents

- product_qna_agent: The user is asking a question about a product. This can be a question about available products, their specifications, user reviews etc.
- shopping_cart_agent: The user is asking to add or remove items from the shopping cart or questions about the current shopping cart.
- warehouse_manager_agent: The user is asking to reserve items from the warehouses or about availability of the items in warehouses.
"""

    template = Template(prompt_template)

    prompt = template.render()

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort="medium",
        use_responses_api=True
    )
    llm_with_tools = llm.bind_tools(
        [FinalAgentResponse, Plan],
        tool_choice="required"
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages
        ]
    )

    final_answer = False
    answer = ""
    # plan = []
    next_agent = ""

    def sanitise_response(response):

        for tool_call in response.tool_calls:
            if tool_call.get("name") == "FinalAgentResponse":
                answer = tool_call.get("args").get("answer")

        return AIMessage(content=answer)

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get("name") == "Plan":
            # plan = response.tool_calls[0].get("args").get("plan")
            next_agent = response.tool_calls[0].get("args").get("next_agent")
            response = None
        else:
            for tool_call in response.tool_calls:
                if tool_call.get("name") == "FinalAgentResponse":
                    final_answer = True
                    answer = tool_call.get("args").get("answer")

                    response = sanitise_response(response)

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "iteration": state.coordinator_agent.iteration + 1,
            "final_answer": final_answer,
            # "plan": plan,
            "next_agent": next_agent
        },
        "answer": answer
    }

In [81]:
results = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x["input"]["messages"],
            coordinator_agent=CoordinatorAgentProperties(
                iteration=0,
                final_answer=False,
                plan=[],
                next_agent=""
            ),
            answer=""
        )
    ),
    data="coordinator-delegation-evaluation-2",
    evaluators=[
        evaluate_coordinator_delegation
    ],
    experiment_prefix="coordinator-delegation[medium reasoning effort]",
    max_concurrency=10,
    num_repetitions=10
)

View the evaluation results for experiment: 'coordinator-delegation[medium reasoning effort]-cb0f9102' at:
https://smith.langchain.com/o/fea55b69-d19d-491c-810a-57660f3b2c4c/datasets/5b0db12b-ec4c-427e-bda6-154f2ea502b0/compare?selectedSessions=2202c861-b80c-4f99-ae0a-bc3a9f8858d1




30it [00:17,  1.70it/s]


### Coordinator Agent V8

In [82]:
class Delegation(BaseModel):
    agent: str = Field(description="The agent to delegate the task to.")
    task: str = Field(description="The task to be performed by the agent.")

class Plan(BaseModel):
    next_agent: str = Field(description="The next agent to invoke")
    # plan: List[Delegation] = Field(description="A list of delegations to agents with tasks to be performed in sequence.")

class FinalAgentResponse(BaseModel):
    answer: str = Field(description="Answer to the question")

class CoordinatorAgentProperties(BaseModel):
    iteration: int = 0
    final_answer: bool = False
    # plan: List[Delegation] = []
    next_agent: str = ""
    
class State(BaseModel):
    messages: Annotated[List[Any], add] = []
    coordinator_agent: CoordinatorAgentProperties = CoordinatorAgentProperties()
    answer: str = ""

In [83]:
@traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-mini"
    }
)
def coordinator_agent(state) -> dict:
    
    prompt_template = """You are a Coordinator Agent as part of a shopping assistant.

## Instructions

- Your role is to delegate work to worker agents in order to solve user queries.
- You should output the next agent to invoke.
- Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and output the next delegation via the `Plan` tool.
- If you do not need to delegate work to any agent, output the final answer via the `FinalAgentResponse` tool.
- Do not route to any agent if the user's query needs clarification or is irrelevant. Do it yourself.
- Only route to the `warehouse_manager_agent` if the user has confirmed that they want to reserve an item and it was already successfully added to the shopping cart.
- Only route to the `shopping_cart_agent` if the user specifically asks about their shopping cart or wants to add or remove items from the shopping cart.
- Do not delegate work to the same agent twice in a row.

## Routing rules

### Multi-intent precedence — highest priority

For requests containing multiple explicit actions, complete unresolved actions in this order:

1. `product_qna_agent`
2. `shopping_cart_agent`
3. `warehouse_manager_agent`

Use the conversation history to determine completed actions.

If `add_to_shopping_cart` succeeded and the user requested reservation, route to `warehouse_manager_agent`.

Only output `FinalAgentResponse` when every explicitly requested action is complete.

Match the user's latest message to one of these patterns:

1. PRODUCT QUESTION — asks about products, specs, reviews, availability, suggestions.
    Examples: "can I get some earphones", "show me laptops", "what watches do you have", "tell me about X", "is X any good", "I'd like to see some X".
    → Route to `product_qna_agent`. After it returns, continue with any explicit cart or reservation actions. Emit `FinalAgentResponse` only when no requested actions remain.

2. EXPLICIT CART INSTRUCTION — uses cart vocabulary.
    Examples: "add to my cart", "remove from my cart", "what's in my cart", "put X in my cart", "delete X from cart".
    → Route to `shopping_cart_agent`. (If items must be discovered first, route to `product_qna_agent` first.)

3. EXPLICIT RESERVATION INSTRUCTION — uses warehouse/reservation vocabulary.
    Examples: "reserve these", "hold these for me", "check warehouse stock for X".
    → Route to `warehouse_manager_agent` ONLY AFTER items are confirmed in the cart.

Default rule: if the user's message does NOT contain explicit cart or warehouse vocabulary, do NOT delegate to those agents. Phrases like "can I get", "I want",
"I'd like", "show me", "I need" are PRODUCT QUESTIONS, not purchase instructions. The user must explicitly say "add", "cart", "reserve", "buy", or "order" before those agents are in scope.

## Available Agents

- product_qna_agent: The user is asking a question about a product. This can be a question about available products, their specifications, user reviews etc.
- shopping_cart_agent: The user is asking to add or remove items from the shopping cart or questions about the current shopping cart.
- warehouse_manager_agent: The user is asking to reserve items from the warehouses or about availability of the items in warehouses.
"""

    template = Template(prompt_template)
    prompt = template.render()

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort="medium",
        use_responses_api=True
    )
    llm_with_tools = llm.bind_tools(
        [FinalAgentResponse, Plan],
        tool_choice="required"
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages
        ]
    )

    final_answer = False
    answer = ""
    next_agent = ""

    def sanitise_response(response):
        for tool_call in response.tool_calls:
            if tool_call.get("name") == "FinalAgentResponse":
                answer = tool_call.get("args").get("answer")

        return AIMessage(content=answer)

    if len(response.tool_calls) > 0:
        if response.tool_calls[0].get("name") == "Plan":
            next_agent = response.tool_calls[0].get("args").get("next_agent")
            response = None
        else:
            for tool_call in response.tool_calls:
                if tool_call.get("name") == "FinalAgentResponse":
                    final_answer = True
                    answer = tool_call.get("args").get("answer")
                    response = sanitise_response(response)

    return {
        "messages": [response] if response else [],
        "coordinator_agent": {
            "iteration": state.coordinator_agent.iteration + 1,
            "final_answer": final_answer,
            "next_agent": next_agent
        },
        "answer": answer
    }

In [84]:
results = ls_client.evaluate(
    lambda x: coordinator_agent(
        State(
            messages=x["input"]["messages"],
            coordinator_agent=CoordinatorAgentProperties(
                iteration=0,
                final_answer=False,
                plan=[],
                next_agent=""
            ),
            answer=""
        )
    ),
    data="coordinator-delegation-evaluation-2",
    evaluators=[
        evaluate_coordinator_delegation
    ],
    experiment_prefix="coordinator-delegation-v8[multi-intent-precedence]",
    max_concurrency=10,
    num_repetitions=10
)

View the evaluation results for experiment: 'coordinator-delegation-v8[multi-intent-precedence]-2dbe3135' at:
https://smith.langchain.com/o/fea55b69-d19d-491c-810a-57660f3b2c4c/datasets/5b0db12b-ec4c-427e-bda6-154f2ea502b0/compare?selectedSessions=3f141afe-0367-4683-9710-f65876ee97aa




30it [01:10,  2.36s/it]
